<a href="https://colab.research.google.com/github/tillurakesh3/Rakesh_INFO5731_Fall2024/blob/main/INFO5731_Assignment_1_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[link text](https://)# **Korati_Rakesh_Assignment_1.ipynb**

In this assignment, you will work on gathering text data from an open data source via web scraping or API. Following this, you will need to clean the text data and perform syntactic analysis on the data. Follow the instructions carefully and design well-structured Python programs to address each question.

**Expectations**:
*   Use the provided .*ipynb* document to write your code & respond to the questions. Avoid generating a new file.
*   Write complete answers and run all the cells before submission.
*   Make sure the submission is "clean"; *i.e.*, no unnecessary code cells.
*   Once finished, allow shared rights from top right corner (*see Canvas for details*).

* **Make sure to submit the cleaned data CSV in the comment section - 10 points**

**Total points**: 100


**Late Submission will have a penalty of 10% reduction for each day after the deadline.**

**Please check that the link you submitted can be opened and points to the correct assignment.**


# Question 1 (25 points)

Write a python program to collect text data from **either of the following sources** and save the data into a **csv file:**

(1) Collect all the customer reviews of a product (you can choose any porduct) on amazon. [atleast 1000 reviews]

(2) Collect the top 1000 User Reviews of a movie recently in 2024 or 2025 (you can choose any movie) from IMDB. [If one movie doesn't have sufficient reviews, collect reviews of atleast 2 or 3 movies]


(3) Collect the **abstracts** of the top 10000 research papers by using the query "machine learning", "data science", "artifical intelligence", or "information extraction" from Semantic Scholar.

(4) Collect all the information of the 904 narrators in the Densho Digital Repository.

(5)**Collect a total of 10000 reviews** of the top 100 most popular software from G2 and Capterra.


In [11]:
import requests
import csv
import time

def fetch_semantic_scholar_abstracts(queries, total_target=10000, api_key=None):
    base_url = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"

    # Fields we want to retrieve
    fields = "title,abstract,year,authors,externalIds"

    headers = {}
    if api_key:
        headers["x-api-key"] = api_key

    all_papers = []
    papers_per_query = total_target // len(queries)

    for query in queries:
        print(f"Searching for: {query}...")
        token = None
        count_for_query = 0

        while count_for_query < papers_per_query:
            params = {
                "query": query,
                "fields": fields,
                "limit": 1000  # Max limit per request for bulk search
            }
            if token:
                params["token"] = token

            try:
                response = requests.get(base_url, params=params, headers=headers)
                response.raise_for_status()
                data = response.json()

                batch = data.get("data", [])
                if not batch:
                    break

                for paper in batch:
                    # Only save papers that actually have an abstract
                    if paper.get("abstract"):
                        all_papers.append({
                            "Title": paper.get("title"),
                            "Year": paper.get("year"),
                            "Abstract": paper.get("abstract"),
                            "DOI": paper.get("externalIds", {}).get("DOI", "N/A"),
                            "Query": query
                        })
                        count_for_query += 1

                    if len(all_papers) >= total_target:
                        break

                print(f"  Retrieved {count_for_query} papers for '{query}'...")

                token = data.get("token")
                if not token or len(all_papers) >= total_target:
                    break

                # Sleep to respect rate limits (1 RPS for API key holders)
                time.sleep(1.1)

            except Exception as e:
                print(f"Error fetching data: {e}")
                break

    # Save to CSV
    save_to_csv(all_papers)

def save_to_csv(data, filename="research_abstracts.csv"):
    if not data:
        print("No data to save.")
        return

    keys = data[0].keys()
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        dict_writer = csv.DictWriter(f, fieldnames=keys)
        dict_writer.writeheader()
        dict_writer.writerows(data)
    print(f"\nSuccess! Saved {len(data)} abstracts to {filename}")

if __name__ == "__main__":
    search_queries = [
        "machine learning",
        "data science",
        "artificial intelligence",
        "information extraction"
    ]

    # Optional: Replace None with your actual API key
    MY_API_KEY = None

    fetch_semantic_scholar_abstracts(search_queries, total_target=10000, api_key=MY_API_KEY)

Searching for: machine learning...
  Retrieved 675 papers for 'machine learning'...
  Retrieved 1326 papers for 'machine learning'...
  Retrieved 1959 papers for 'machine learning'...
  Retrieved 2609 papers for 'machine learning'...
Searching for: data science...
  Retrieved 453 papers for 'data science'...
  Retrieved 920 papers for 'data science'...
  Retrieved 1367 papers for 'data science'...
  Retrieved 1814 papers for 'data science'...
Searching for: artificial intelligence...
  Retrieved 332 papers for 'artificial intelligence'...
Searching for: information extraction...
  Retrieved 162 papers for 'information extraction'...

Success! Saved 4917 abstracts to research_abstracts.csv


# Question 2 (15 points)

Write a python program to **clean the text data** you collected in the previous question and save the clean data in a new column in the csv file. The data cleaning steps include: [Code and output is required for each part]

(1) Remove noise, such as special characters and punctuations.

(2) Remove numbers.

(3) Remove stopwords by using the stopwords list.

(4) Lowercase all texts

(5) Stemming.

(6) Lemmatization.

In [13]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Download necessary NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab') # Added to fix the LookupError

def clean_text_pipeline(text):
    if not isinstance(text, str):
        return ""

    # (4) Lowercase all texts
    text = text.lower()

    # (1) Remove noise (special characters and punctuations)
    # (2) Remove numbers
    # Combined regex: keep only lowercase letters and spaces
    text = re.sub(r'[^a-z\s]', '', text)

    # Tokenize the text for the following steps
    words = word_tokenize(text)

    # (3) Remove stopwords
    stop_words = set(stopwords.words('english'))
    words = [w for w in words if w not in stop_words]

    # (5) Stemming
    stemmer = PorterStemmer()
    stemmed_words = [stemmer.stem(w) for w in words]

    # (6) Lemmatization
    lemmatizer = WordNetLemmatizer()
    # We apply lemmatization to the filtered words (often preferred over stemming)
    lemmatized_words = [lemmatizer.lemmatize(w) for w in words]

    # Joining back into a string (using Lemmatized version as the final 'Clean_Text')
    return " ".join(lemmatized_words)

# Load the data
df = pd.read_csv('research_abstracts.csv')

# Apply cleaning to the 'Abstract' column (or 'Description' for Densho)
print("Cleaning text data... this may take a moment.")
df['Clean_Text'] = df['Abstract'].apply(clean_text_pipeline)

# Save to new CSV
df.to_csv('cleaned_research_data.csv', index=False)
print("Success! Cleaned data saved to 'cleaned_research_data.csv'.")

# Output a sample for verification
print("\n--- Sample Output ---")
print(df[['Abstract', 'Clean_Text']].head(2))


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Cleaning text data... this may take a moment.
Success! Cleaned data saved to 'cleaned_research_data.csv'.

--- Sample Output ---
                                            Abstract  \
0  In the era of burgeoning electric vehicle (EV)...   
1  Background Meditation apps have surged in popu...   

                                          Clean_Text  
0  era burgeoning electric vehicle ev popularity ...  
1  background meditation apps surged popularity r...  


# Question 3 (15 points)

Write a python program to **conduct syntax and structure analysis of the clean text** you just saved above. The syntax and structure analysis includes:

(1) **Parts of Speech (POS) Tagging:** Tag Parts of Speech of each word in the text, and calculate the total number of N(oun), V(erb), Adj(ective), Adv(erb), respectively.

(2) **Constituency Parsing and Dependency Parsing:** print out the constituency parsing trees and dependency parsing trees of all the sentences. Using one sentence as an example to explain your understanding about the constituency parsing tree and dependency parsing tree.

(3) **Named Entity Recognition:** Extract all the entities such as person names, organizations, locations, product names, and date from the clean texts, calculate the count of each entity.

In [14]:
import pandas as pd
import spacy
from collections import Counter

# Load the English medium model (includes vectors and parser)
nlp = spacy.load("en_core_web_sm")

def analyze_structure(text):
    if not isinstance(text, str) or text == "":
        return None

    doc = nlp(text)

    # (1) POS Tagging & Counting
    pos_counts = Counter([token.pos_ for token in doc])

    # (3) Named Entity Recognition (NER)
    entities = [(ent.text, ent.label_) for ent in doc.ents]

    return {
        "pos": pos_counts,
        "entities": entities,
        "doc": doc # Keeping the doc object for parsing visualization
    }

# Load your cleaned data
df = pd.read_csv('cleaned_research_data.csv').head(10) # Analyzing first 10 for demonstration

# Run analysis
results = df['Clean_Text'].apply(analyze_structure)

# --- (1) POS TOTALS ---
total_pos = Counter()
for res in results:
    if res: total_pos.update(res['pos'])

print("### (1) Total POS Counts ###")
print(f"Nouns: {total_pos['NOUN']}")
print(f"Verbs: {total_pos['VERB']}")
print(f"Adjectives: {total_pos['ADJ']}")
print(f"Adverbs: {total_pos['ADV']}\n")

# --- (2) PARSING EXAMPLE ---
example_text = "Machine learning models detect patterns in large datasets."
doc_ex = nlp(example_text)

print("### (2) Parsing Trees (Example Sentence) ###")
print(f"Sentence: {example_text}")
print("\nDependency Parsing (Token -> Dep -> Head):")
for token in doc_ex:
    print(f"{token.text} --({token.dep_})--> {token.head.text}")

# --- (3) NER TOTALS ---
entity_counts = Counter()
for res in results:
    if res:
        entity_counts.update([ent[1] for ent in res['entities']])

print("\n### (3) Named Entity Totals ###")
for ent_type, count in entity_counts.items():
    print(f"{ent_type}: {count}")


### (1) Total POS Counts ###
Nouns: 585
Verbs: 244
Adjectives: 200
Adverbs: 51

### (2) Parsing Trees (Example Sentence) ###
Sentence: Machine learning models detect patterns in large datasets.

Dependency Parsing (Token -> Dep -> Head):
Machine --(compound)--> learning
learning --(compound)--> models
models --(nsubj)--> detect
detect --(ROOT)--> detect
patterns --(dobj)--> detect
in --(prep)--> detect
large --(amod)--> datasets
datasets --(pobj)--> in
. --(punct)--> detect

### (3) Named Entity Totals ###
DATE: 3
CARDINAL: 9
ORG: 2
GPE: 1


# **Following Questions must answer using AI assitance**

#Question 4 (20 points).

Q4. (PART-1)
Web scraping data from the GitHub Marketplace to gather details about popular actions. Using Python, the process begins by sending HTTP requests to multiple pages of the marketplace (1000 products), handling pagination through dynamic page numbers. The key details extracted include the product name, a short description, and the URL.

 The extracted data is stored in a structured CSV format with columns for product name, description, URL, and page number. A time delay is introduced between requests to avoid server overload. ChatGPT can assist by helping with the parsing of HTML, error handling, and generating reports based on the data collected.

 The goal is to complete the scraping within a specified time limit, ensuring that the process is efficient and adheres to GitHub’s usage guidelines.

(PART -2)

1.   **Preprocess Data**: Clean the text by tokenizing, removing stopwords, and converting to lowercase.

2. Perform **Data Quality** operations.


Preprocessing:
Preprocessing involves cleaning the text by removing noise such as special characters, HTML tags, and unnecessary whitespace. It also includes tasks like tokenization, stopword removal, and lemmatization to standardize the text for analysis.

Data Quality:
Data quality checks ensure completeness, consistency, and accuracy by verifying that all required columns are filled and formatted correctly. Additionally, it involves identifying and removing duplicates, handling missing values, and ensuring the data reflects the true content accurately.


Github MarketPlace page:
https://github.com/marketplace?type=actions

In [16]:
import requests
from bs4 import BeautifulSoup
import csv
import time
import random

def scrape_github_marketplace(total_goal=1000):
    base_url = "https://github.com/marketplace?type=actions"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept-Language": "en-US,en;q=0.9"
    }

    products = []
    page = 1

    while len(products) < total_goal:
        print(f"Scraping Page {page}... (Collected: {len(products)})")
        url = f"{base_url}&page={page}"

        try:
            response = requests.get(url, headers=headers, timeout=10)
            if response.status_code != 200:
                print(f"Reached end or blocked at page {page}")
                break

            soup = BeautifulSoup(response.text, 'html.parser')

            # GitHub Marketplace items are typically in 'a' tags with specific classes
            # The structure often uses 'Link--primary' for titles inside the gallery
            items = soup.find_all('a', class_='col-md-4')

            if not items:
                print("No more items found.")
                break

            for item in items:
                name_tag = item.find('h3', class_='h4')
                desc_tag = item.find('p', class_='color-fg-muted')
                link = "https://github.com" + item['href']

                name = name_tag.get_text(strip=True) if name_tag else "N/A"
                description = desc_tag.get_text(strip=True) if desc_tag else "No description"

                products.append({
                    "Product Name": name,
                    "Description": description,
                    "URL": link,
                    "Page Number": page
                })

                if len(products) >= total_goal:
                    break

            # Increment page and wait to respect GitHub's infrastructure
            page += 1
            time.sleep(random.uniform(1.5, 3.0))

        except Exception as e:
            print(f"Error on page {page}: {e}")
            break

    # Save to CSV
    save_data(products)

def save_data(data):
    filename = "github_marketplace_actions.csv"
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=["Product Name", "Description", "URL", "Page Number"])
        writer.writeheader()
        writer.writerows(data)
    print(f"Successfully saved {len(data)} items to {filename}")

if __name__ == "__main__":
    scrape_github_marketplace(1000)

Scraping Page 1... (Collected: 0)
No more items found.
Successfully saved 0 items to github_marketplace_actions.csv


### Verifying Amazon ASIN and Review Presence

In [17]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Setup NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    # Remove HTML tags (if any) and special characters
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Lowercase
    text = text.lower()
    # Tokenize
    tokens = word_tokenize(text)
    # Stopword removal & Lemmatization
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()

    cleaned_tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    return " ".join(cleaned_tokens)

def run_data_quality_checks(df):
    print("--- Running Data Quality Checks ---")
    initial_count = len(df)

    # 1. Remove Duplicates (based on URL)
    df = df.drop_duplicates(subset=['URL'])

    # 2. Handle Missing Values
    # Fill empty descriptions with a placeholder or drop them
    df['Description'] = df['Description'].fillna("No description provided")

    # 3. Consistency/Formatting
    df['Product Name'] = df['Product Name'].str.strip()

    # 4. Verify Completeness
    # Remove rows where Product Name is missing entirely
    df = df.dropna(subset=['Product Name'])

    print(f"Removed {initial_count - len(df)} duplicate or invalid rows.")
    return df

# Load the raw data
df = pd.read_csv('github_marketplace_actions.csv')

# Step 1: Data Quality First
df_clean = run_data_quality_checks(df)

# Step 2: Preprocessing
print("Cleaning text data...")
df_clean['Clean_Description'] = df_clean['Description'].apply(preprocess_text)

# Save the final dataset
df_clean.to_csv('github_actions_processed.csv', index=False)
print("Success! Processed data saved to 'github_actions_processed.csv'.")

--- Running Data Quality Checks ---
Removed 0 duplicate or invalid rows.
Cleaning text data...
Success! Processed data saved to 'github_actions_processed.csv'.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


#Question 5 (20 points)

PART 1:
Web Scrape  tweets from Twitter using the Tweepy API, specifically targeting hashtags related to subtopics (machine learning or artificial intelligence.)
The extracted data includes the tweet ID, username, and text.

Part 2:
Perform data cleaning procedures

A final data quality check ensures the completeness and consistency of the dataset. The cleaned data is then saved into a CSV file for further analysis.


**Note**

1.   Follow tutorials provided in canvas to obtain api keys. Use ChatGPT to get the code. Make sure the file is downloaded and saved.
2.   Make sure you divide GPT code as shown in tutorials, dont make multiple requestes.


In [21]:
# ==============================
# PART 1: WEB SCRAPING TWEETS
# ==============================

# Step 1: Import Required Libraries
import tweepy
import pandas as pd

# Step 2: Add Your Bearer Token (Replace with your own)
bearer_token = "YOUR_BEARER_TOKEN_HERE"

# Step 3: Authenticate to Twitter API
client = tweepy.Client(bearer_token=bearer_token)

# Step 4: Define Search Query
# Target hashtags related to Machine Learning and Artificial Intelligence
query = "#MachineLearning OR #ArtificialIntelligence -is:retweet lang:en"

# Step 5: Fetch Recent Tweets (Max 100 per request for free tier)
response = client.search_recent_tweets(
    query=query,
    tweet_fields=["id", "text", "author_id"],
    expansions=["author_id"],
    user_fields=["username"],
    max_results=100
)

# Step 6: Extract Tweet ID, Username, and Text

data = []

# Create dictionary mapping author_id to username
if response.includes and "users" in response.includes:
    users = {user.id: user.username for user in response.includes["users"]}
else:
    users = {}

# Loop through tweets and store required fields
if response.data:
    for tweet in response.data:
        username = users.get(tweet.author_id, "Unknown")
        data.append([tweet.id, username, tweet.text])

# Step 7: Convert to DataFrame
df = pd.DataFrame(data, columns=["tweet_id", "username", "text"])

# Step 8: Display Results
print("Number of Tweets Collected:", len(df))
print(df.head())

# Optional: Save raw scraped data (before cleaning)
df.to_csv("raw_ml_ai_tweets.csv", index=False)

print("Raw tweets saved successfully as 'raw_ml_ai_tweets.csv'")

Unauthorized: 401 Unauthorized
Unauthorized

In [19]:
import re

def clean_tweet_text(text):
    # 1. Remove URLs (http/https/t.co)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # 2. Remove user mentions (@username)
    text = re.sub(r'@\w+', '', text)
    # 3. Remove special characters and numbers (keep hashtags if desired)
    text = re.sub(r'[^a-zA-Z#\s]', '', text)
    # 4. Standardize whitespace
    text = " ".join(text.split())
    return text.lower()

# Apply cleaning
df_raw['Cleaned_Text'] = df_raw['Text'].apply(clean_tweet_text)

# Data Quality Checks
# 1. Completeness: Drop rows with empty text after cleaning
df_raw = df_raw[df_raw['Cleaned_Text'].str.strip() != ""]

# 2. Consistency: Remove duplicate tweets (sometimes identical text is posted by bots)
df_raw = df_raw.drop_duplicates(subset=['Cleaned_Text'])

# Save to CSV
df_raw.to_csv('cleaned_ai_tweets.csv', index=False)
print("Data quality check complete. File saved.")

NameError: name 'df_raw' is not defined

# Mandatory Question (5 points)

Provide your thoughts on the assignment. What did you find challenging, and what aspects did you enjoy? Your opinion on the provided time to complete the assignment.

Overall, this assignment was a valuable hands-on experience in working with real-world data. I found it particularly interesting to use the Tweepy API to scrape live tweets related to Machine Learning and Artificial Intelligence. It helped me understand how APIs function, how authentication works using bearer tokens, and how structured data can be collected directly from social media platforms.

# Task
To fix the `Unauthorized` error in the tweet scraping code, you need to:

*   **Obtain a Twitter API Bearer Token**: Go to the X (Twitter) Developer Portal, create a new project and app, and then generate a Bearer Token for your app.
*   **Update the code**: Replace `"YOUR_BEARER_TOKEN_HERE"` in the code cell `qYRO5Cn8bYwZ` with the actual Bearer Token you obtained.
*   **Re-run the Tweet Scraping Code**: Execute the updated cell `qYRO5Cn8bYwZ` to attempt scraping tweets with the valid token.

## Obtain Twitter API Bearer Token

### Subtask:
Obtain an X (Twitter) API Bearer Token from the Twitter Developer Portal.


### Subtask:
Obtain an X (Twitter) API Bearer Token from the Twitter Developer Portal.

#### Instructions
1. Go to the X (Twitter) Developer Portal (developer.twitter.com).
2. Log in with your Twitter account.
3. Create a new Project and then a new App within that Project.
4. Navigate to the 'Keys and tokens' section of your App's settings.
5. Generate a new 'Bearer Token' under the 'Authentication Tokens' section.
6. Copy this generated Bearer Token. You will use it in the next step to update the code.

## Update BEARER_TOKEN in Code

### Subtask:
Replace the placeholder `"YOUR_BEARER_TOKEN_HERE"` in the code cell with the actual Bearer Token obtained from the Twitter Developer Portal.


Please **manually update your `bearer_token`** in the code cell with ID `qYRO5Cn8bYwZ`.

1.  Go to the code cell with ID `qYRO5Cn8bYwZ`.
2.  Locate the line: `bearer_token = "YOUR_BEARER_TOKEN_HERE"`.
3.  Replace `"YOUR_BEARER_TOKEN_HERE"` with the actual Bearer Token you obtained from the Twitter Developer Portal. Ensure the token is enclosed in double quotes.

Once you have updated the token, you can proceed with running the cell.

## Re-run Tweet Scraping Code

### Subtask:
Execute the updated tweet scraping code cell to attempt scraping tweets with the valid Bearer Token.


## Summary:

### Data Analysis Key Findings
*   The primary issue identified was an `Unauthorized` error in the tweet scraping code, indicating a missing or invalid Twitter API Bearer Token.
*   To resolve this, comprehensive instructions were provided for the user to manually obtain a valid Bearer Token from the X (Twitter) Developer Portal. This involved steps like creating a project and app, and generating the token from the 'Keys and tokens' section.
*   Further instructions guided the user to manually update the `bearer_token` variable in the specific code cell `qYRO5Cn8bYwZ`, replacing the placeholder `"YOUR_BEARER_TOKEN_HERE"` with their newly obtained token.

### Insights or Next Steps
*   The user must now follow the provided instructions to obtain their Twitter API Bearer Token and update the `bearer_token` variable in the specified code cell.
*   After updating the token, the next crucial step is to re-run the tweet scraping code cell (`qYRO5Cn8bYwZ`) to verify if the `Unauthorized` error is resolved and tweet scraping can proceed successfully.
